# Notebook 2 : Diamants 💎

In [ ]:
# Décommenter la ligne suivante pour installer les dépendances
# %pip install jupyter_bokeh nbconvert panel watchfiles

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import panel as pn
import plotly.graph_objects as go
import plotly.express as px

pn.extension("plotly", "tabulator")

Les données utilisées pour cette application correspondent à des caractéristiques de 53940 diamants :

- `price` : prix en dollars américains,
- `carat` : masse du diamant,
- `cut` : qualité de la taille,
- `color` : couleur allant de `D` (meilleure) à `J` (pire),
- `clarity` : code de clarté du diamant,
- `x`, `y`, `z` : dimensions du diamant.

## Préliminaires

1. Écrire une fonction `get_data` sans argument qui retourne les données contenues dans le fichier `data/diamonds.csv` sous la forme d'un DataFrame Pandas. Cette fontion a vocation à être appelée plusieurs fois avec le même résultat attendu, elle pourra donc être décorée avec `pn.cache`.

In [ ]:
@pn.cache
def get_data():
    data_path = "data/diamonds.csv"
    print(f"Charge les données depuis {data_path}")
    return pd.read_csv(data_path)

2. Utiliser la méthode `value_counts` de Pandas pour compter les effectifs de diamants selon la variable `cut`.

In [ ]:
diamonds = get_data()
cut_counts = diamonds.cut.value_counts()
cut_counts

 3. Utiliser la fonction `pie` de Matplotlib (voir [la documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.pie.html)) pour visualiser les effectifs de la question précédente sous la forme d'un camembert.

In [ ]:
fig, ax = plt.subplots()
ax.pie(cut_counts, labels=cut_counts.index)
plt.show()

4. Utiliser la fonction `scatter` de Matplotlib (voir [la documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html)) pour afficher le nuage de points associé à la variable `price` en fonction de la variable `carat`. Colorer les points en fonction de la variable `color` et faire apparaître la légende correspondante sur le graphique.

In [ ]:
fig, ax = plt.subplots()

# Affichage d'un nuage de point par couleur
for color, df in diamonds.groupby("color"):
    ax.scatter(df.carat, df.price, label=color)

# Affichage de la légende en bas à droite
plt.legend(loc="lower right")

# Bonus : noms des axes
plt.xlabel("Carat")
plt.ylabel("Prix")

plt.show()

5. Utiliser la fonction `scatter` de Plotly (voir [la documentation](https://plotly.com/python-api-reference/generated/plotly.express.scatter)) pour afficher le nuage de points associé à la variable `price` en fonction de la variable `carat`. Colorer les points en fonction de la variable `color`. Noter que la légende est directement intégrée au graphique et que ce dernier offre plusieurs possibilités d'interaction (zoom, filtre, ...).

In [ ]:
fig = px.scatter(x=diamonds.carat, y=diamonds.price, color=diamonds.color)

# Bonus : noms des axes
fig.update_layout(xaxis_title="Carat", yaxis_title= "Prix")

## Filtre des diamants

6. Créer un selecteur multiple `colors` de type `MultiChoice` (voir [la documentation](https://panel.holoviz.org/reference/widgets/MultiChoice.html)) pour sélectionner certaines couleurs des diamants du jeu de données (toutes les couleurs devront être sélectionnées par défaut). Afficher l'objet dans le notebook avec un nom (`name`) associé. Quel est le contenu et le type de l'attribut `value` de cet objet ?

In [ ]:
diamonds = get_data()

all_colors = sorted(diamonds.color.unique().tolist())
colors = pn.widgets.MultiChoice(
    name="Couleur",
    value=all_colors,
    options=all_colors,
    # Bonus : un message d'information
    placeholder="Sélectionner les couleurs",
)

print(type(colors.value)) # Type list
print(colors.value) # Liste des chaînes de caractères sélectionnées

colors

7. Écrire une fonction `get_colored_diamonds` avec un argument `colors` contenant une liste de couleurs de diamants qui retourne les données obtenues avec `get_data` filtrées selon cette liste de couleurs. Cette fontion a vocation à être appelée plusieurs fois avec le même résultat attendu, elle pourra donc être décorée avec `pn.cache`.

In [ ]:
@pn.cache
def get_colored_diamonds(colors):
    data = get_data()
    return data[data.color.isin(colors)]

8. À l'aide de `pn.bind`, lier la fonction `get_colored_diamonds` à la valeur du widget `colors` de la question 6 pour le paramètre `colors`. Placer le résultat dans un widget `Tabulator` (voir [la documentation](https://panel.holoviz.org/reference/widgets/Tabulator.html)) qui sera affiché dans le notebook. Masquer l'index dans le `Tabulator` et désactiver l'édition (paramètre `disabled`). Limiter le nombre de lignes par page à `10`. Faire varier la valeur du widget pour visualiser l'interaction.

In [ ]:
pn.widgets.Tabulator(
    pn.bind(get_colored_diamonds, colors=colors),
    # Masque l'index
    show_index=False,
    # Désactive l'édition
    disabled=True,
    # Limite la page à 10 lignes
    page_size=10,
    # Bonus : désactive la sélection
    selectable=False,
)

## Fonctions graphiques

9. Écrire une fonction `get_pie` avec un argument `colors` qui retourne l'objet graphique Matplotlib du camembert de la question 3 pour les données filtrées par couleur selon `colors`. Penser à fermer la figure avec `plt.close` avant de la retourner pour éviter les fuites de mémoire.

In [ ]:
def get_pie(colors):
    diamonds = get_colored_diamonds(colors) # Données filtrées
    cut_counts = diamonds.cut.value_counts() # Compte par catégorie
    
    fig, ax = plt.subplots()
    ax.pie(cut_counts, labels=cut_counts.index)

    # Bonus : fond transparent
    fig.patch.set_alpha(0.0)
    
    plt.close(fig) # Évite les fuites de mémoire
    return fig

10. Écrire une fonction `get_scatter_matplotlib` avec les arguments `colors` et `title` qui retourne l'objet graphique Matplotlib du nuage de points de la question 4 pour les données filtrées par couleur selon `colors` avec un titre donné par `title`. Penser à fermer la figure avec `plt.close` avant de la retourner pour éviter les fuites de mémoire.

In [ ]:
def get_scatter_matplotlib(colors, title):
    diamonds = get_colored_diamonds(colors) # Données filtrées

    fig, ax = plt.subplots()
    for color, df in diamonds.groupby("color"):
        ax.scatter(df.carat, df.price, label=color)
    
    plt.title(title) # Ajout d'un titre
    plt.legend(loc="lower right")
    plt.xlabel("Carat")
    plt.ylabel("Prix")

    # Bonus : fond transparent
    ax.patch.set_alpha(0.0)
    fig.patch.set_alpha(0.0)

    plt.close(fig)
    return fig

11. Écrire une fonction `get_scatter_plotly` avec les arguments `colors` et `title` qui retourne l'objet graphique Plotly du nuage de points de la question 5 pour les données filtrées par couleur selon `colors` avec un titre donné par `title`.

In [ ]:
def get_scatter_plotly(colors, title):
    diamonds = get_colored_diamonds(colors) # Données filtrées

    fig = px.scatter(x=diamonds.carat, y=diamonds.price, color=diamonds.color)
    fig.update_layout(
        xaxis_title="Carat",
        yaxis_title= "Prix",
        # Ajout d'un titre
        title=title,
    )

    return fig

## Application web

Nous pouvons maintenant mettre en place une application web qui pourra être démarrée avec la commande suivante (l'option `--allow-websocket-origin` n'est nécessaire que dans Onyxia) :
```{bash}
panel serve --autoreload --show --allow-websocket-origin=$(echo $VSCODE_PROXY_URI | cut -d '/' -f 3) notebooks/02_diamants.ipynb
```

Les questions suivantes ont pour objet d'enrichir l'application au fur et à mesure. Il ne faut donc pas recréer une nouvelle application pour chaque question mais faire évoluer le code étape par étape. Il peut être utile d'ajouter de nouvelles cellules de code si besoin.

12. Mettre en forme une application à l'aide du modèle `FastListTemplate` (voir [la documentation](https://panel.holoviz.org/reference/templates/FastListTemplate.html)) avec le widget `colors` et un widget `title` de type `TextInput` (voir [la documentation](https://panel.holoviz.org/reference/widgets/TextInput.html)) dans la barre latérale et une zone principale telle que :
  - le widget `Tabulator` des données filtrées se trouve en haut à gauche,
  - le *pane* contenant le camembert obtenu avec la fonction `get_pie` se trouve en haut à droite,
  - le *pane* contenant le graphique Matplotlib obtenu avec la fonction `get_scatter_matplotlib` occupe la partie inférieure.

13. Utiliser les paramètres `height` et `sizing_mode` du *panes* contenant le camembert pour adapter sa taille. Remarquer l'aspect peu satisfaisant de l'affichage des graphiques Matplotlib.

14. Remplacer le *pane* du nuage de points Matplotlib par un *pane* adapté à l'objet graphique retourné par `get_scatter_plotly`. Remarquer comment l'aspect du graphique Plotly s'adapte à la taille de la fenêtre.

15. Ajouter un *pane* de type `Markdown` (voir [la documentation](https://panel.holoviz.org/reference/panes/Markdown.html)) en haut à droite de l'application contenant un texte mis en forme pour afficher les moyennes et les variances des variables quantitatives du jeu de données filtré selon `colors`.

In [ ]:
# Question 15
def format_statistics(df, name):
    result = f"- Variable `{name}` :\n"
    result += f"  - Moyenne : {df[name].mean():.2f}\n"
    result += f"  - Écart-type : {df[name].std():.2f}\n"
    return result

def get_statistics(colors):
    diamonds = get_colored_diamonds(colors) # Données filtrées
    result = "## Quelques statistiques\n"
    for column in ["carat", "depth", "table", "price"]:
        result += format_statistics(diamonds, column)
    return result

statistics = pn.bind(get_statistics, colors=colors)

In [ ]:
# Widget pour le titre du graphique
title = pn.widgets.TextInput(name="Titre", placeholder="Titre du graphique")

# Fonctions liées aux widgets
colored_diamonds = pn.bind(get_colored_diamonds, colors=colors)
pie = pn.bind(get_pie, colors=colors)
scatter_matplotlib = pn.bind(get_scatter_matplotlib, colors=colors, title=title)
scatter_plotly = pn.bind(get_scatter_plotly, colors=colors, title=title) # Question 14

pn.template.FastListTemplate(
    title="Diamants 💎",
    sidebar=[colors, title],
    main=[
            pn.Row(
                pn.widgets.Tabulator(
                    colored_diamonds,
                    show_index=False,
                    disabled=True,
                    page_size=10,
                    selectable=False,
                ),
                pn.pane.Matplotlib(
                    pie,
                    height=360, # Question 13
                    sizing_mode="stretch_width", # Question 13
                ),
                pn.pane.Markdown(statistics), # Question 15
            ),
            pn.Row(
                pn.pane.Matplotlib(scatter_matplotlib),
                # Question 14 (Commenter le pane Matplotlib)
                pn.pane.Plotly(scatter_plotly, sizing_mode="stretch_width"),
            )
    ],
).servable()